# 02 — Trajectory Analysis

Load saved episode logs and analyse trajectory properties:
- Token count distribution across modes (raw / llm_summary / compressor)
- Step count per episode
- Tool call redundancy rate
- Compression ratio (raw tokens → compressed tokens)

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
from optimized_llm_planning_memory.utils.episode_io import list_episodes

episodes = list_episodes('../outputs/episodes')
print(f'Loaded {len(episodes)} episodes')

In [ ]:
# Step count distribution
step_counts = [ep.total_steps for ep in episodes]
if step_counts:
    import statistics
    print(f'Steps — mean: {statistics.mean(step_counts):.1f}, ')
    print(f'         min: {min(step_counts)}, max: {max(step_counts)}')
else:
    print('No episodes loaded. Run scripts/run_episode.py first.')

In [ ]:
# Tool call redundancy
for ep in episodes[:3]:
    total = sum(s.call_count for s in ep.tool_stats)
    redundant = sum(s.redundant_call_count for s in ep.tool_stats)
    print(f'Episode {ep.episode_id[:8]}: {total} calls, {redundant} redundant')

In [ ]:
# Compression ratio
for ep in episodes[:3]:
    if ep.compressed_states:
        raw_len = len(ep.trajectory.to_text()) if hasattr(ep.trajectory, 'to_text') else 0
        comp_len = sum(cs.token_count or 0 for cs in ep.compressed_states)
        if raw_len > 0:
            print(f'Episode {ep.episode_id[:8]}: raw≈{raw_len} chars, compressed≈{comp_len} tokens')